## Prompting with LangChain

---



In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Replace the path with the full path to your .env file
load_dotenv('/content/drive/MyDrive/llms_for_production/.env')

# Access the keys as environment variables
openai_key = os.getenv("OPENAI_API_KEY")
deeplake_token = os.getenv("DEEPLAKE_TOKEN")

print("Keys loaded:", bool(openai_key), bool(deeplake_token))

client=OpenAI(api_key=os.environ['OPENAI_API_KEY'])


Keys loaded: True True


In [3]:
 #Install langchain
!pip -q install langchain langchain-openai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate


In [6]:
# Initialize the model
llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0)
template=""" Answer the question based on the context below.If the question cannot be answered using the information provided, answer with "I don't known"
context: Quantum computing is an emerging field that leverages quantum mechanics to solve complex problems faster than classical computers.
...
Question: {query}
Answer: """
prompt_template=PromptTemplate(template=template,input_variables=['query'])

#Create the LLMChain for the prompt
chain = prompt_template | llm



In [7]:
#Set the query you want to ask
input_data={"query":"What is the difference between classical and quantum computing?"}
#Invoke the chain
reponse=chain.invoke(input_data)
print(reponse.content)


Quantum computing leverages quantum mechanics to solve complex problems faster than classical computers.


# FewShotTemplate with an ExampleSelector


In [11]:
from langchain_core.prompts import FewShotPromptTemplate
from langchain_openai import ChatOpenAI

#Initialise the model
llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0)

#Build the template
examples=[
    {"animal": "lion", "habitat": "savanna"},
    {"animal": "polar bear", "habitat": "Arctic ice"},
    {"animal": "elephant", "habitat": "African grasslands"}
]

example_template="""
Animal={animal}
Habitat={habitat}
"""
example_prompt=PromptTemplate(input_variables=["animal","habitat"],template=example_template)


dynamic_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Identify the habitat of the given animal",
    suffix="Animal: {input}\nHabitat:",
    input_variables=["input"],
    example_separator="\n\n",
)

#Create LLM chain for the prompt
chain=dynamic_prompt | llm

#Run che LLMChain with input_data
input_data={"input":"tiger"}
response=chain.invoke(input_data)
print(response.content)




tropical rainforests, grasslands, and mangrove swamps


In [15]:
#Save your promptemplate in local file system in JSON or YAML format
import json

with open("dynamic_prompt.json", "w") as f:
    json.dump(dynamic_prompt.dict(), f, indent=4)

In [18]:
#Load the saved prompt
with open("dynamic_prompt.json", "r") as f:
    loaded_dynamic_prompt = json.load(f)

In [22]:
examples = [
    {
 "query": "How do I become a better programmer?",
 "answer": "Try talking to a rubber duck; it works wonders."
    },
     {
 "query": "Why is the sky blue?",
 "answer": "It's nature's way of preventing eye strain."
    }
]

example_template="""
User: {query}
AI: {answer}
"""

example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template=example_template
)

prefix = """The following are excerpts from conversations with an AI
assistant. The assistant is typically sarcastic and witty, producing
creative and funny responses to users' questions. Here are some
examples:

"""

suffix = """
User: {query}
AI: """
few_shot_prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n\n"
)

# Create the LLMChain for the few_shot_prompt_template
chain=few_shot_prompt_template | llm
# Run the LLMChain with input_data
input_data = {"query": "How can I learn quantum computing?"}
response = chain.invoke(input_data)
print(response)


content='Step 1: Watch every episode of Rick and Morty. Step 2: Realize you still have no idea what quantum computing is. Step 3: Enroll in a course.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 102, 'total_tokens': 141, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVEmulTKHF4eGgrSYUx2kLn7DWHdk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d95f5-7439-7793-a693-e83cb355be6a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 102, 'output_tokens': 39, 'total_tokens': 141, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
